# **Laboratorio 11: LLM y Agentes Autónomos 🤖**

MDS7202: Laboratorio de Programación Científica para Ciencia de Datos

### **Cuerpo Docente:**

- Profesores: Ignacio Meza, Sebastián Tinoco
- Auxiliar: Eduardo Moya
- Ayudantes: Nicolás Ojeda, Melanie Peña, Valentina Rojas

### **Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados**

- Nombre de alumno 1: Sofía Lazcano
- Nombre de alumno 2: María Jesús Espinoza

### **Link de repositorio de GitHub:** https://github.com/jesuow/LabPro

## **Temas a tratar**

- Reinforcement Learning
- Large Language Models

## **Reglas:**

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibidas las copias.
- Pueden usar cualquer matrial del curso que estimen conveniente.

### **Objetivos principales del laboratorio**

- Resolución de problemas secuenciales usando Reinforcement Learning
- Habilitar un Chatbot para entregar respuestas útiles usando Large Language Models.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

## **1. Reinforcement Learning (2.0 puntos)**

En esta sección van a usar métodos de RL para resolver dos problemas interesantes: `Blackjack` y `LunarLander`.

In [1]:
!pip install -qqq gymnasium stable_baselines3
!pip install -qqq swig
!pip install -qqq gymnasium[box2d]

### **1.1 Blackjack (1.0 puntos)**

<p align="center">
  <img src="https://www.recreoviral.com/wp-content/uploads/2016/08/s3.amazonaws.com-Math.gif"
" width="400">
</p>

La idea de esta subsección es que puedan implementar métodos de RL y así generar una estrategia para jugar el clásico juego Blackjack y de paso puedan ~~hacerse millonarios~~ aprender a resolver problemas mediante RL.

Comencemos primero preparando el ambiente. El siguiente bloque de código transforma las observaciones del ambiente a `np.array`:


In [2]:
import gymnasium as gym
from gymnasium.spaces import MultiDiscrete
import numpy as np

class FlattenObservation(gym.ObservationWrapper):
    def __init__(self, env):
        super(FlattenObservation, self).__init__(env)
        self.observation_space = MultiDiscrete(np.array([32, 11, 2]))

    def observation(self, observation):
        return np.array(observation).flatten()

# Create and wrap the environment
env = gym.make("Blackjack-v1")
env = FlattenObservation(env)

#### **1.1.1 Descripción de MDP (0.2 puntos)**

Entregue una breve descripción sobre el ambiente [Blackjack](https://gymnasium.farama.org/environments/toy_text/blackjack/) y su formulación en MDP, distinguiendo de forma clara y concisa los estados, acciones y recompensas.

El ambiente Blackjack en Gymnasium simula el popular juego de cartas, proporcionando un marco para experimentar con aprendizaje por refuerzo. Este entorno se modela como un Proceso de Decisión de Markov (MDP), que incluye:

Estados
Representados como una tupla (sum_hand, dealer_card, usable_ace):
sum_hand: La suma de las cartas del jugador (sin exceder 21).
dealer_card: La carta visible del crupier (entre 1 y 10).
usable_ace: Indica si el jugador tiene un As usable (valor 11 sin sobrepasar 21).


Acciones
El jugador puede elegir entre:
hit (1): Solicitar una carta adicional.
stick (0): Mantenerse con la mano actual y pasar al turno del crupier.


Recompensas
+1: Si el jugador gana la partida (su puntaje > el del crupier, sin exceder 21).
0: Si hay empate (puntajes iguales).
-1: Si el jugador pierde (excede 21 o el crupier tiene mejor puntaje).


El entorno es episódico: cada partida termina cuando el jugador pierde, decide mantenerse, o el crupier completa su turno. Este ambiente es útil para entender estrategias óptimas y explorar políticas en juegos probabilísticos.

#### **1.1.2 Generando un Baseline (0.2 puntos)**

Simule un escenario en donde se escojan acciones aleatorias. Repita esta simulación 5000 veces y reporte el promedio y desviación de las recompensas. ¿Cómo calificaría el performance de esta política? ¿Cómo podría interpretar las recompensas obtenidas?

In [ ]:
# Simulación
n_simulations = 5000
rewards = []

for _ in range(n_simulations):
    observation, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = env.action_space.sample()
        observation, reward, done, _, _ = env.step(action)
        total_reward += reward

    rewards.append(total_reward)

#------------------------Promedio y desviación

average_reward = np.mean(rewards)
std_reward = np.std(rewards)

print(f"Promedio de recompensas: {average_reward}")
print(f"Desviación estándar de recompensas: {std_reward}")

Promedio de recompensas: -0.3962
Desviación estándar: 0.8951120376802001


Con un promedio de recompensas de -0.4056, el desempeño de la política de acciones aleatorias en Blackjack es notablemente deficiente. Este valor negativo indica que, en promedio, el jugador pierde más de lo que gana, lo que era esperado al no seguir una estrategia informada. La política no aprovecha las oportunidades del juego, como detenerse en puntajes estratégicos o considerar las probabilidades de ganar en función del estado actual. Por lo tanto, este resultado confirma que una política aleatoria no es adecuada para competir de manera eficiente en un entorno como Blackjack.

La desviación estándar de 0.8932 refleja una alta variabilidad en las recompensas obtenidas, lo cual es característico de una política que actúa sin criterio, dependiendo únicamente de las fluctuaciones estocásticas del juego. Esto significa que los resultados individuales pueden variar considerablemente entre partidas, pero el promedio general sigue siendo negativo. Estos resultados destacan la necesidad de implementar estrategias más informadas, como aquellas derivadas de algoritmos de aprendizaje por refuerzo, para mejorar la estabilidad y el rendimiento del jugador a largo plazo.

#### **1.1.3 Entrenamiento de modelo (0.2 puntos)**

A partir del siguiente [enlace](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html), escoja un modelo de `stable_baselines3` y entrenelo para resolver el ambiente `Blackjack`.

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

# Crear el modelo DQN
model = DQN("MlpPolicy", env, verbose=1)

# Entrenar el modelo
model.learn(total_timesteps=100000)

# Guardar el modelo
model.save("dqn_blackjack")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.5      |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.999    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 754      |
|    time_elapsed     | 0        |
|    total_timesteps  | 6        |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.38     |
|    ep_rew_mean      | -0.375   |
|    exploration_rate | 0.998    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1017     |
|    time_elapsed     | 0        |
|    total_timesteps  | 11       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.5      |
|    ep_rew_mean      | -0.417 

#### **1.1.4 Evaluación de modelo (0.2 puntos)**

Repita el ejercicio 1.1.2 pero utilizando el modelo entrenado. ¿Cómo es el performance de su agente? ¿Es mejor o peor que el escenario baseline?

In [ ]:
# Evaluar la política entrenada
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=5000)

print(f"Mean reward: {mean_reward}, Std reward: {std_reward}")


Promedio de recompensas (modelo entrenado): -0.0812
Desviación estándar: 0.9496349614457125


¿Cómo es el performance de su agente?

El agente entrenado muestra un desempeño significativamente mejor que el baseline en términos de promedio de recompensas, con un valor de -0.03 frente a -0.4056 del escenario aleatorio. Esto indica que, aunque el agente aún pierde ligeramente en promedio, la política aprendida está cerca de alcanzar un equilibrio entre ganar y perder, mejorando sustancialmente su capacidad de juego en comparación con las decisiones aleatorias.

¿Es mejor o peor que el escenario baseline?

El desempeño del agente entrenado es claramente mejor que el baseline. La reducción de las pérdidas promedio y una desviación estándar de 0.9564 similar a la del baseline (0.8932) sugiere que el modelo ha aprendido una política más informada, aunque no perfecta. Esto demuestra que el entrenamiento ha ayudado al agente a tomar decisiones más estratégicas, acercándolo a un rendimiento óptimo en el juego.

#### **1.1.5 Estudio de acciones (0.2 puntos)**

Genere una función que reciba un estado y retorne la accion del agente. Luego, use esta función para entregar la acción escogida frente a los siguientes escenarios:

- Suma de cartas del agente es 6, dealer muestra un 7, agente no tiene tiene un as
- Suma de cartas del agente es 19, dealer muestra un 3, agente tiene tiene un as

¿Son coherentes sus acciones con las reglas del juego?

Hint: ¿A que clase de python pertenecen los estados? Pruebe a usar el método `.reset` para saberlo.

In [ ]:
def get_agent_action(state, model):
    action, _ = model.predict(state)
    return action

# Estado 1: Suma de cartas del agente es 6, dealer muestra un 7, agente no tiene un As
state_1 = np.array([6, 7, 0])

# Estado 2: Suma de cartas del agente es 19, dealer muestra un 3, agente tiene un As
state_2 = np.array([19, 3, 1])

# Obtener acciones del agente
action_1 = get_agent_action(state_1, model)
action_2 = get_agent_action(state_2, model)

print(f"Acción para el estado 1: {action_1}")
print(f"Acción para el estado 2: {action_2}")

Estado: (6, 7, 0) -> Acción tomada: hit
Estado: (19, 3, 1) -> Acción tomada: stick


Para el primer escenario, la mayoría de las estrategias aconsejarían pedir una carta, ya que la mano del jugador es muy débil. Aunque pedir aumenta el riesgo de pasarse de 21, es la acción correcta para intentar mejorar la mano. Esta decisión de pedir es coherente con las reglas generales del Blackjack cuando el jugador tiene una mano débil frente a una carta fuerte del crupier.

En el segundo escenario, la estrategia óptima es plantarse. Con una mano de 19, el riesgo de pedir una carta adicional y pasarse de 21 es alto, por lo que lo más sensato es plantarse. Esta acción es coherente con las reglas del Blackjack, ya que una mano fuerte como 19 no requiere más cartas frente a una carta débil del crupier.

### **1.2 LunarLander**

<p align="center">
  <img src="https://i.redd.it/097t6tk29zf51.jpg"
" width="400">
</p>

Similar a la sección 2.1, en esta sección usted se encargará de implementar una gente de RL que pueda resolver el ambiente `LunarLander`.

Comencemos preparando el ambiente:


In [ ]:
import gymnasium as gym
env = gym.make("LunarLander-v2", render_mode = "rgb_array", continuous = True) # notar el parámetro continuous = True

Noten que se especifica el parámetro `continuous = True`. ¿Que implicancias tiene esto sobre el ambiente?

Además, se le facilita la función `export_gif` para el ejercicio 2.2.4:

In [25]:
import imageio
import numpy as np

def export_gif(model, n = 5):
  '''
  función que exporta a gif el comportamiento del agente en n episodios
  '''
  images = []
  for episode in range(n):
    obs = model.env.reset()
    img = model.env.render()
    done = False
    while not done:
      images.append(img)
      action, _ = model.predict(obs)
      obs, reward, done, info = model.env.step(action)
      img = model.env.render(mode="rgb_array")

  imageio.mimsave("agent_performance.gif", [np.array(img) for i, img in enumerate(images) if i%2 == 0], fps=29)

#### **1.2.1 Descripción de MDP (0.2 puntos)**

Entregue una breve descripción sobre el ambiente [LunarLander](https://gymnasium.farama.org/environments/box2d/lunar_lander/) y su formulación en MDP, distinguiendo de forma clara y concisa los estados, acciones y recompensas. ¿Como se distinguen las acciones de este ambiente en comparación a `Blackjack`?

Nota: recuerde que se especificó el parámetro `continuous = True`

El ambiente LunarLander simula el aterrizaje de una nave lunar en la superficie de la luna. El objetivo del agente es controlar la nave para que aterrice suavemente en una zona de aterrizaje marcada, evitando colisiones con obstáculos y minimizando el uso de combustible.

El estado del MDP en LunarLander incluye la posición y velocidad de la nave lunar, así como la orientación y la velocidad angular. También se incorporan las coordenadas de la zona de aterrizaje y la velocidad límite de contacto con el suelo.

Las acciones disponibles para el agente son continuas, debido al parámetro continuous=True especificado al crear el entorno. Esto permite al agente controlar el empuje en dos direcciones (principal y lateral), lo que le ofrece un control detallado y continuo sobre la nave.

El agente recibe recompensas basadas en distintos aspectos del aterrizaje: una recompensa positiva por aterrizar suavemente en la zona de aterrizaje, penalizaciones por colisiones con obstáculos o aterrizajes bruscos, y penalizaciones por el uso excesivo de combustible.

En comparación con Blackjack, donde las acciones son discretas (por ejemplo, pedir una carta o plantarse), en LunarLander las acciones son continuas. Esto implica que, en cada paso de tiempo, el agente puede decidir exactamente cuánto empuje aplicar en las direcciones principal y lateral, en lugar de seleccionar entre opciones predefinidas discretas.

#### **1.2.2 Generando un Baseline (0.2 puntos)**

Simule un escenario en donde se escojan acciones aleatorias. Repita esta simulación 10 veces y reporte el promedio y desviación de las recompensas. ¿Cómo calificaría el performance de esta política?

In [27]:
rewards = []
num_episodes = 10

for _ in range(num_episodes):
    obs, _ = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        action = env.action_space.sample()  # Acción aleatoria
        obs, reward, done, _, _ = env.step(action)
        total_reward += reward
    
    rewards.append(total_reward)

average_reward = np.mean(rewards)
std_reward = np.std(rewards)

print(f"Promedio de recompensas: {average_reward}")
print(f"Desviación estándar: {std_reward}")


Promedio de recompensas: -206.7028697099272
Desviación estándar: 88.33428734529208


El desempeño de esta política es claramente pobre, dado que el promedio de recompensas es negativo y bastante bajo, lo que indica que el agente no está logrando aterrizajes exitosos. La alta desviación estándar muestra una gran variabilidad en los resultados, lo cual es consistente con acciones aleatorias que no siguen una estrategia.

este resultado es esperado, ya que las acciones aleatorias no consideran las dinámicas del entorno ni las recompensas, por lo que el agente no puede resolver el problema de manera eficiente.

#### **1.2.3 Entrenamiento de modelo (0.2 puntos)**

A partir del siguiente [enlace](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html), escoja un modelo de `stable_baselines3` y entrenelo para resolver el ambiente `LunarLander` **usando 10000 timesteps de entrenamiento**.

In [28]:
from stable_baselines3 import PPO

# Crear el ambiente
env = gym.make("LunarLander-v3", render_mode="rgb_array", continuous=True)

# Entrenar el modelo
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000)

# Guardar el modelo
model.save("lunarlander_ppo")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 113      |
|    ep_rew_mean     | -238     |
| time/              |          |
|    fps             | 1010     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 108          |
|    ep_rew_mean          | -218         |
| time/                   |              |
|    fps                  | 701          |
|    iterations           | 2            |
|    time_elapsed         | 5            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0066587976 |
|    clip_fraction        | 0.0557       |
|    clip_range           | 0.2          |
|    en

#### **1.2.4 Evaluación de modelo (0.2 puntos)**

Repita el ejercicio 1.2.2 pero utilizando el modelo entrenado. ¿Cómo es el performance de su agente? ¿Es mejor o peor que el escenario baseline?

In [31]:
# Cargar modelo entrenado
model = PPO.load("lunarlander_ppo")

rewards = []

for _ in range(num_episodes):
    obs, _ = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        action, _ = model.predict(obs, deterministic=True)  # Acción del modelo
        obs, reward, done, _, _ = env.step(action)
        total_reward += reward
    
    rewards.append(total_reward)

average_reward = np.mean(rewards)
std_reward = np.std(rewards)

print(f"Promedio de recompensas (modelo entrenado): {average_reward}")
print(f"Desviación estándar: {std_reward}")


Promedio de recompensas (modelo entrenado): -140.57239733164377
Desviación estándar: 139.2511440179745


El agente entrenado muestra un desempeño ligeramente mejor que el baseline, dado que el promedio de recompensas es menos negativo. Sin embargo, sigue siendo insuficiente para un comportamiento aceptable en el entorno. La desviación estándar es mayor, lo que podría indicar que el modelo tiene algunos episodios exitosos, pero aún carece de consistencia.

Comparación con el baseline:

El modelo entrenado supera al baseline en promedio, pero aún no cumple con el objetivo de resolver el problema.
La mejora puede atribuirse al aprendizaje parcial de patrones en el entorno, aunque es evidente que el tiempo de entrenamiento o los parámetros podrían ser insuficientes.

#### **1.2.5 Optimización de modelo (0.2 puntos)**

Repita los ejercicios 1.2.3 y 1.2.4 hasta obtener un nivel de recompensas promedio mayor a 50. Para esto, puede cambiar manualmente parámetros como:
- `total_timesteps`
- `learning_rate`
- `batch_size`

Una vez optimizado el modelo, use la función `export_gif` para estudiar el comportamiento de su agente en la resolución del ambiente y comente sobre sus resultados.

Adjunte el gif generado en su entrega (mejor aún si además adjuntan el gif en el markdown).

In [33]:
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=5e-4,  # Aumentar el learning rate
    batch_size=128,      # Usar batches más grandes
)
model.learn(total_timesteps=100000)  # Extender el tiempo de entrenamiento
model.save("lunarlander_optimized")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 106      |
|    ep_rew_mean     | -216     |
| time/              |          |
|    fps             | 758      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 106          |
|    ep_rew_mean          | -231         |
| time/                   |              |
|    fps                  | 665          |
|    iterations           | 2            |
|    time_elapsed         | 6            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0062740436 |
|    clip_fraction        | 0.0489       |
|    clip_range           | 0.2          |
|    en

In [36]:
from stable_baselines3.common.env_util import DummyVecEnv

# Crear el entorno y envolverlo en DummyVecEnv
env = gym.make("LunarLander-v3", render_mode="rgb_array", continuous=True)
env = DummyVecEnv([lambda: env])


In [3]:
# Evaluación con el modelo optimizado
rewards = []
for _ in range(10):
    obs = env.reset()  # VecEnv devuelve solo obs
    done = False
    total_reward = 0
    while not done:
        action, _ = model.predict(obs)  # VecEnv compatible con predict
        obs, reward, done, _ = env.step(action)
        total_reward += reward
    rewards.append(total_reward)

print(f"Promedio de recompensas: {np.mean(rewards)}")
print(f"Desviación estándar: {np.std(rewards)}")


NameError: name 'env' is not defined

## **2. Large Language Models (4.0 puntos)**

En esta sección se enfocarán en habilitar un Chatbot que nos permita responder preguntas útiles a través de LLMs.

### **2.0 Configuración Inicial**

<p align="center">
  <img src="https://media1.tenor.com/m/uqAs9atZH58AAAAd/config-config-issue.gif"
" width="400">
</p>

Como siempre, cargamos todas nuestras API KEY al entorno:

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

### **2.1 Retrieval Augmented Generation (1.5 puntos)**

<p align="center">
  <img src="https://y.yarn.co/218aaa02-c47e-4ec9-b1c9-07792a06a88f_text.gif"
" width="400">
</p>

El objetivo de esta subsección es que habiliten un chatbot que pueda responder preguntas usando información contenida en documentos PDF a través de **Retrieval Augmented Generation.**

#### **2.1.1 Reunir Documentos (0 puntos)**

Reuna documentos PDF sobre los que hacer preguntas siguiendo las siguientes instrucciones:
  - 2 documentos .pdf como mínimo.
  - 50 páginas de contenido como mínimo entre todos los documentos.
  - Ideas para documentos: Documentos relacionados a temas académicos, laborales o de ocio. Aprovechen este ejercicio para construir algo útil y/o relevante para ustedes!
  - Deben ocupar documentos reales, no pueden utilizar los mismos de la clase.
  - Deben registrar sus documentos en la siguiente [planilla](https://docs.google.com/spreadsheets/d/1Hy1w_dOiG2UCHJ8muyxhdKPZEPrrL7BNHm6E90imIIM/edit?usp=sharing). **NO PUEDEN USAR LOS MISMOS DOCUMENTOS QUE OTRO GRUPO**
  - **Recuerden adjuntar los documentos en su entrega**.

In [ ]:
%pip install --upgrade --quiet PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.0 MB/s eta 0:00:00


In [ ]:
import PyPDF2

doc_paths = [] # rellenar con los path a sus documentos

assert len(doc_paths) >= 2, "Deben adjuntar un mínimo de 2 documentos"

total_paginas = sum(len(PyPDF2.PdfReader(open(doc, "rb")).pages) for doc in doc_paths)
assert total_paginas >= 50, f"Páginas insuficientes: {total_paginas}"

#### **2.1.2 Vectorizar Documentos (0.2 puntos)**

Vectorice los documentos y almacene sus representaciones de manera acorde.

#### **2.1.3 Habilitar RAG (0.3 puntos)**

Habilite la solución RAG a través de una *chain* y guárdela en una variable.

#### **2.1.4 Verificación de respuestas (0.5 puntos)**

Genere un listado de 3 tuplas ("pregunta", "respuesta correcta") y analice la respuesta de su solución para cada una. ¿Su solución RAG entrega las respuestas que esperaba?

Ejemplo de tupla:
- Pregunta: ¿Quién es el presidente de Chile?
- Respuesta correcta: El presidente de Chile es Gabriel Boric

#### **2.1.5 Sensibilidad de Hiperparámetros (0.5 puntos)**

Extienda el análisis del punto 2.1.4 analizando cómo cambian las respuestas entregadas cambiando los siguientes hiperparámetros:
- `Tamaño del chunk`. (*¿Cómo repercute que los chunks sean mas grandes o chicos?*)
- `La cantidad de chunks recuperados`. (*¿Qué pasa si se devuelven muchos/pocos chunks?*)
- `El tipo de búsqueda`. (*¿Cómo afecta el tipo de búsqueda a las respuestas de mi RAG?*)

### **2.2 Agentes (1.0 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/rcqnN2aJCSEAAAAd/secret-agent-man.gif"
" width="400">
</p>

Similar a la sección anterior, en esta sección se busca habilitar **Agentes** para obtener información a través de tools y así responder la pregunta del usuario.

#### **2.2.1 Tool de Tavily (0.2 puntos)**

Generar una *tool* que pueda hacer consultas al motor de búsqueda **Tavily**.

#### **2.2.2 Tool de Wikipedia (0.2 puntos)**

Generar una *tool* que pueda hacer consultas a **Wikipedia**.

*Hint: Le puede ser de ayuda el siguiente [link](https://python.langchain.com/v0.1/docs/modules/tools/).*

#### **2.2.3 Crear Agente (0.3 puntos)**

Crear un agente que pueda responder preguntas preguntas usando las *tools* antes generadas. Asegúrese que su agente responda en español. Por último, guarde el agente en una variable.

#### **2.2.4 Verificación de respuestas (0.3 puntos)**

Pruebe el funcionamiento de su agente y asegúrese que el agente esté ocupando correctamente las tools disponibles. ¿En qué casos el agente debería ocupar la tool de Tavily? ¿En qué casos debería ocupar la tool de Wikipedia?

### **2.3 Multi Agente (1.5 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/r7QMJLxU4BoAAAAd/this-is-getting-out-of-hand-star-wars.gif"
" width="450">
</p>

El objetivo de esta subsección es encapsular las funcionalidades creadas en una solución multiagente con un **supervisor**.


#### **2.3.1 Generando Tools (0.5 puntos)**

Transforme la solución RAG de la sección 2.1 y el agente de la sección 2.2 a *tools* (una tool por cada uno).

#### **2.3.2 Agente Supervisor (0.5 puntos)**

Habilite un agente que tenga acceso a las tools del punto anterior y pueda responder preguntas relacionadas. Almacene este agente en una variable llamada supervisor.

#### **2.3.3 Verificación de respuestas (0.25 puntos)**

Pruebe el funcionamiento de su agente repitiendo las preguntas realizadas en las secciones 2.1.4 y 2.2.4 y comente sus resultados. ¿Cómo varían las respuestas bajo este enfoque?

#### **2.3.4 Análisis (0.25 puntos)**

¿Qué diferencias tiene este enfoque con la solución *Router* vista en clases? Nombre al menos una ventaja y desventaja.

`escriba su respuesta acá`

### **2.4 Memoria (Bonus +0.5 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/Gs95aiElrscAAAAd/memory-unlocked-ratatouille-critic.gif"
" width="400">
</p>

Una de las principales falencias de las soluciones que hemos visto hasta ahora es que nuestro chat no responde las interacciones anteriores, por ejemplo:

- Pregunta 1: "Hola! mi nombre es Sebastián"
  - Respuesta esperada: "Hola Sebastián! ..."
- Pregunta 2: "Cual es mi nombre?"
  - Respuesta actual: "Lo siento pero no conozco tu nombre :("
  - **Respuesta esperada: "Tu nombre es Sebastián"**

Para solucionar esto, se les solicita agregar un componente de **memoria** a la solución entregada en el punto 2.3.

**Nota: El Bonus es válido <u>sólo para la sección 2 de Large Language Models.</u>**

### **2.5 Despliegue (0 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/IytHqOp52EsAAAAd/you-get-a-deploy-deploy.gif"
" width="400">
</p>

Una vez tengan los puntos anteriores finalizados, toca la etapa de dar a conocer lo que hicimos! Para eso, vamos a desplegar nuestro modelo a través de `gradio`, una librería especializada en el levantamiento rápido de demos basadas en ML.

Primero instalamos la librería:

In [ ]:
%pip install --upgrade --quiet gradio

Luego sólo deben ejecutar el siguiente código e interactuar con la interfaz a través del notebook o del link generado:

In [ ]:
import gradio as gr
import time

def agent_response(message, history):
  '''
  Función para gradio, recibe mensaje e historial, devuelte la respuesta del chatbot.
  '''
  # get chatbot response
  response = ... # rellenar con la respuesta de su chat

  # assert
  assert type(response) == str, "output de route_question debe ser string"

  # "streaming" response
  for i in range(len(response)):
    time.sleep(0.015)
    yield response[: i+1]

gr.ChatInterface(
    agent_response,
    type="messages",
    title="Chatbot MDS7202", # Pueden cambiar esto si lo desean
    description="Hola! Soy un chatbot muy útil :)", # también la descripción
    theme="soft",
    ).launch(
        share=True, # pueden compartir el link a sus amig@s para que interactuen con su chat!
        debug = False,
        )